# 评价段 SEC 索引与原始 XML 身份校验

零网络解析 15 对已归档的 SOXX NPORT filing index 与 primary XML；逐项核查 accession、申报日、报告期、系列类别、30 条 EC。仅来源身份，不计算值或收益。

In [1]:
import json,hashlib,re,xml.etree.ElementTree as ET
from pathlib import Path
from bs4 import BeautifulSoup
root=Path.cwd();card=root/'research/stage1_soxx_2023_2026_nport_original_source_20261023';plan=json.loads((card/'plan.json').read_text());transport=json.loads((card/'transport_result.json').read_text());receipts=json.loads((card/'capture_receipts.json').read_text())
def sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert transport['status']=='fifteen_held_index_and_XML_pairs_archived_parse_pending' and transport['nas_remote_and_scoped_restore_verified'] and len(receipts)==30
raw=root/'data/snapshots/stage1_soxx_2023_2026_nport_original_source_20261023';receipt_by={(x['accession'],x['kind']):x for x in receipts};rows=[];errors=[]
for spec in plan['exact_filings']:
 a=spec['accession'];ix=raw/(a+'.index.html');xp=raw/(a+'.XML.xml');ri=receipt_by[(a,'index')];rx=receipt_by[(a,'XML')]
 assert sha(ix)==ri['raw_sha256'] and sha(xp)==rx['raw_sha256']
 try:
  soup=BeautifulSoup(ix.read_bytes(),'html.parser');sec=soup.select_one('#secNum');form=soup.select_one('#formName');assert sec and a in sec.get_text(' ',strip=True) and form and spec['form'] in form.get_text(' ',strip=True)
  info={}
  for group in soup.select('div.formContent div.formGrouping'):
   children=group.find_all('div',recursive=False)
   for j,node in enumerate(children[:-1]):
    if 'infoHead' in (node.get('class') or []):info[node.get_text(' ',strip=True)]=children[j+1].get_text(' ',strip=True)
  filed=info.get('Filing Date');accepted=info.get('Accepted');period=info.get('Period of Report')
  assert filed==spec['filed_from_legacy_manifest'] and period==spec['report_period'] and re.fullmatch(r'20\d\d-\d\d-\d\d \d\d:\d\d:\d\d',accepted)
  div=soup.select_one('#seriesDiv');assert div
  sr=[[q.get_text(' ',strip=True) for q in tr.find_all('td',recursive=False)] for tr in div.select('tr')]
  sp=[i for i,x in enumerate(sr) if x and x[0]=='Series S000004354'];cp=[i for i,x in enumerate(sr) if x and x[0]=='Class/Contract C000012084']
  assert len(sp)==len(cp)==1 and cp[0]==sp[0]+1 and 'SOXX' in sr[cp[0]] and 'iShares' in ' '.join(sr[sp[0]])
  docs=[]
  for tr in soup.select('table.tableFile tr'):
   cells=tr.find_all('td',recursive=False)
   if len(cells)<5:continue
   link=cells[2].find('a')
   if link:docs.append({'document':link.get_text(' ',strip=True),'href':link.get('href'),'type':cells[3].get_text(' ',strip=True)})
  primary=[x for x in docs if x['type']==spec['form'] and x['document']=='primary_doc.xml' and x['href'].endswith('/primary_doc.xml')]
  assert len(primary)==1 and primary[0]['href'].split('?')[0] in spec['primary_xml_url']
  xml=ET.fromstring(xp.read_bytes());local=lambda t:t.rsplit('}',1)[-1]
  def val(key):return sorted({(q.text or '').strip() for q in xml.iter() if local(q.tag)==key and (q.text or '').strip()})
  assert val('cik')==['0001100663'] and val('seriesId')==['S000004354'] and val('classId')==['C000012084'] and val('repPdDate')==[period]
  rows_all=[q for q in xml.iter() if local(q.tag)=='invstOrSec']
  def field(e,k):return next(((q.text or '').strip() for q in e if local(q.tag)==k),'')
  cats={k:sum(field(e,'assetCat')==k for e in rows_all) for k in ('EC','DE','STIV')}
  assert cats['EC']==30 and sum(cats.values())==len(rows_all)
  rows.append({'accession':a,'form':spec['form'],'filed':filed,'accepted_display':accepted,'accepted_display_timezone':'SEC_page_unverified','report_period':period,'series_id':'S000004354','class_id':'C000012084','ticker_from_index':'SOXX','index_sha256':ri['raw_sha256'],'original_XML_sha256':rx['raw_sha256'],'primary_XML_href':primary[0]['href'],'asset_categories':cats,'original_equity_value_qualified':False})
 except Exception as exc:errors.append({'accession':a,'error':type(exc).__name__+':'+str(exc)[:160]})
result={'status':'fifteen_original_held_NPORT_index_XML_identities_qualified_values_pending' if not errors and len(rows)==15 else 'held_NPORT_source_identity_stop','source_plan_sha256':sha(card/'plan.json'),'source_NAS_archive_sha256':transport['nas_archive_sha256'],'source_NAS_scoped_restore_verified':True,'expected_filings':15,'qualified_filings':len(rows),'original_base_filings':sum(x['form']=='NPORT-P' for x in rows),'later_amendments':sum(x['form']=='NPORT-P/A' for x in rows),'errors':errors,'rows':rows,'market_outcome_reads':0,'model_fits':0,'native_accounts':0}
(card/'identity_result.json').write_text(json.dumps(result,indent=2)+'\n')
print({'status':result['status'],'qualified':len(rows),'errors':errors,'last':rows[-1] if rows else None})


{'status': 'fifteen_original_held_NPORT_index_XML_identities_qualified_values_pending', 'qualified': 15, 'errors': [], 'last': {'accession': '0002071691-26-015796', 'form': 'NPORT-P/A', 'filed': '2026-07-13', 'accepted_display': '2026-07-13 10:48:21', 'accepted_display_timezone': 'SEC_page_unverified', 'report_period': '2025-09-30', 'series_id': 'S000004354', 'class_id': 'C000012084', 'ticker_from_index': 'SOXX', 'index_sha256': 'ca123ac37d3c2d55ae5be536b7faf2e6981e642a8b3063b244e6e681b58c8482', 'original_XML_sha256': '6b884f242db23c0f31de65b5728c774ca1f29514ed47fb4eced6741685fdeff5', 'primary_XML_href': '/Archives/edgar/data/1100663/000207169126015796/primary_doc.xml', 'asset_categories': {'EC': 30, 'DE': 1, 'STIV': 2}, 'original_equity_value_qualified': False}}


## 中文结论

原件身份结果见 `identity_result.json`。迟到修订的值、时钟和有效报告顺序后续单独判断。